In [1]:
# 🚀 EOQ Clásico en Google Colab — Versión Inicial
# ================================================

# 🔹 Paso 1: Instalar dependencias necesarias
!pip install pandas openpyxl --quiet

# 🔹 Paso 2: Importar librerías
import pandas as pd
import numpy as np

# 🔹 Paso 3: Cargar archivo subido
from google.colab import files

print("📎 Sube el archivo 'retail_store_inventory.csv':")
uploaded = files.upload()
file_path = list(uploaded.keys())[0]  # Obtener nombre del archivo

# 🔹 Paso 4: Leer CSV
df = pd.read_csv(file_path)

# 🔹 Paso 5: Seleccionar un producto específico
producto_id = "P0001"
df_producto = df[df["Product ID"] == producto_id]

if df_producto.empty:
    raise ValueError(f"No se encontraron datos para el producto '{producto_id}'")

# 🔹 Paso 6: Parámetros EOQ
D = df_producto["Units Sold"].sum()   # Demanda anual estimada
S = 50                                # Coste por pedido (€)
precio_medio = df_producto["Price"].mean()
H = 0.2 * precio_medio                # Coste mantenimiento anual por unidad

# 🔹 Paso 7: Calcular EOQ y métricas
EOQ = np.sqrt((2 * D * S) / H)
n_pedidos = D / EOQ
CT_total = (D / EOQ) * S + (EOQ / 2) * H

# 🔹 Paso 8: Mostrar resultados
print("\n📊 RESULTADOS EOQ CLÁSICO\n--------------------------")
print(f"📦 Producto analizado:       {producto_id}")
print(f"📈 Demanda anual estimada:   {round(D):,} uds")
print(f"💰 Coste por pedido (S):     {S:.2f} €")
print(f"🏷️ Coste mantener unidad:    {H:.2f} €/año")
print(f"📦 Cantidad óptima a pedir:  {round(EOQ):,} uds")
print(f"🔁 Pedidos por año:          {round(n_pedidos, 2)}")
print(f"💸 Coste total anual (CT):   {round(CT_total, 2):,} €")


📎 Sube el archivo 'retail_store_inventory.csv':


Saving retail_store_inventory.csv to retail_store_inventory (1).csv

📊 RESULTADOS EOQ CLÁSICO
--------------------------
📦 Producto analizado:       P0001
📈 Demanda anual estimada:   498,061 uds
💰 Coste por pedido (S):     50.00 €
🏷️ Coste mantener unidad:    10.91 €/año
📦 Cantidad óptima a pedir:  2,137 uds
🔁 Pedidos por año:          233.1
💸 Coste total anual (CT):   23,310.04 €


In [2]:
# 🚀 EOQ CON DESCUENTOS POR CANTIDAD – Portfolio Ready

# Paso 1: Librerías
import pandas as pd
import numpy as np
from google.colab import files

# Paso 2: Subida del archivo
print("📎 Sube el archivo 'retail_store_inventory.csv':")
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
df = pd.read_csv(file_path)

# Paso 3: Elegir producto
producto_id = "P0001"
df_producto = df[df["Product ID"] == producto_id]

if df_producto.empty:
    raise ValueError(f"No se encontraron datos para el producto '{producto_id}'")

# Paso 4: Parámetros
D = df_producto["Units Sold"].sum()  # Demanda anual
S = 50  # Coste por pedido

# Definimos los tramos con descuentos por cantidad
descuentos = [
    {"min": 0, "max": 999,   "precio": 50},
    {"min": 1000, "max": 1999, "precio": 48},
    {"min": 2000, "max": float("inf"), "precio": 46},
]

# Paso 5: Evaluación por tramo
resultados = []

for tramo in descuentos:
    P = tramo["precio"]
    H = 0.2 * P  # Coste de mantener unidad
    Q = np.sqrt((2 * D * S) / H)

    # Ajustamos Q al mínimo del tramo si cae fuera
    if Q < tramo["min"]:
        Q = tramo["min"]
    elif Q > tramo["max"]:
        Q = tramo["max"]

    n_pedidos = D / Q
    CT = (n_pedidos * S) + (Q / 2 * H) + (D * P)

    resultados.append({
        "Tramo Pedido": f"{tramo['min']} - {tramo['max'] if tramo['max'] != float('inf') else '∞'}",
        "Precio Unitario (€)": P,
        "Coste Mantener (H)": round(H, 2),
        "Cantidad Pedida (Q)": round(Q),
        "Pedidos/año": round(n_pedidos, 2),
        "Coste Total Anual (€)": round(CT, 2)
    })

# Paso 6: Mostrar como tabla profesional
df_resultados = pd.DataFrame(resultados)
df_resultados.sort_values(by="Coste Total Anual (€)", inplace=True)

print("\n📊 Comparativa EOQ con descuentos por cantidad:\n")
from tabulate import tabulate
print(tabulate(df_resultados, headers="keys", tablefmt="github"))


📎 Sube el archivo 'retail_store_inventory.csv':


Saving retail_store_inventory.csv to retail_store_inventory (2).csv

📊 Comparativa EOQ con descuentos por cantidad:

|    | Tramo Pedido   |   Precio Unitario (€) |   Coste Mantener (H) |   Cantidad Pedida (Q) |   Pedidos/año |   Coste Total Anual (€) |
|----|----------------|-----------------------|----------------------|-----------------------|---------------|-------------------------|
|  2 | 2000 - ∞       |                    46 |                  9.2 |                  2327 |        214.06 |             2.29322e+07 |
|  1 | 1000 - 1999    |                    48 |                  9.6 |                  1999 |        249.16 |             2.3929e+07  |
|  0 | 0 - 999        |                    50 |                 10   |                   999 |        498.56 |             2.4933e+07  |


In [3]:
# 🚀 EOQ AVANZADO CON DESCUENTOS Y OPTIMIZACIÓN CONTINUA

# 📦 Paso 1: Librerías
!pip install pandas tabulate scipy --quiet

import pandas as pd
import numpy as np
from google.colab import files
from scipy.optimize import minimize_scalar
from tabulate import tabulate

# 📂 Paso 2: Subida del archivo
print("📎 Sube el archivo 'retail_store_inventory.csv':")
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
df = pd.read_csv(file_path)

# 🎯 Paso 3: Elegir producto y calcular demanda anual
producto_id = "P0001"
df_producto = df[df["Product ID"] == producto_id]

if df_producto.empty:
    raise ValueError(f"No se encontraron datos para el producto '{producto_id}'")

D = df_producto["Units Sold"].sum()     # Demanda anual
S = 50                                  # Coste por pedido (€)

# 🧾 Paso 4: Tramos con descuentos por cantidad
descuentos = [
    {"min": 0, "max": 999,   "precio": 50},
    {"min": 1000, "max": 1999, "precio": 48},
    {"min": 2000, "max": float("inf"), "precio": 46},
]

resultados = []

# 🧮 Paso 5: Análisis EOQ por tramo de descuento
for tramo in descuentos:
    P = tramo["precio"]
    H = 0.2 * P  # Coste de mantener unidad/año
    Q = np.sqrt((2 * D * S) / H)

    if Q < tramo["min"]:
        Q = tramo["min"]
    elif Q > tramo["max"]:
        Q = tramo["max"]

    n_pedidos = D / Q
    CT = (n_pedidos * S) + (Q / 2 * H) + (D * P)

    resultados.append({
        "Tramo Pedido": f"{tramo['min']} - {tramo['max'] if tramo['max'] != float('inf') else '∞'}",
        "Precio Unitario (€)": P,
        "Coste Mantener (H)": round(H, 2),
        "Cantidad Pedida (Q)": round(Q),
        "Pedidos/año": round(n_pedidos, 2),
        "Coste Total Anual (€)": round(CT, 2)
    })

df_resultados = pd.DataFrame(resultados)
df_resultados.sort_values(by="Coste Total Anual (€)", inplace=True)

# 📊 Mostrar tabla EOQ por tramo
print("\n📊 Comparativa EOQ con descuentos por cantidad:\n")
print(tabulate(df_resultados, headers="keys", tablefmt="github"))

# 🔧 Paso 6: Optimización continua sobre el tramo más barato (precio 46 €)
precio_opt = 46
H_opt = 0.2 * precio_opt

def coste_total(Q, D, S, H, P):
    Q = max(Q, 1e-3)  # Evitar división por cero
    return (D / Q) * S + (Q / 2) * H + D * P

res = minimize_scalar(coste_total, bounds=(1, D), method='bounded', args=(D, S, H_opt, precio_opt))

# ✅ Resultados de la optimización
Q_opt = round(res.x)
CT_opt = round(res.fun, 2)
n_pedidos_opt = round(D / Q_opt, 2)

print("\n🧠 Optimización continua EOQ con precio mínimo (46 €):\n")
print(f"📦 Cantidad óptima a pedir: {Q_opt} uds")
print(f"🔁 Pedidos por año: {n_pedidos_opt}")
print(f"💸 Coste total anual estimado: {CT_opt:,.2f} €")


📎 Sube el archivo 'retail_store_inventory.csv':


Saving retail_store_inventory.csv to retail_store_inventory (3).csv

📊 Comparativa EOQ con descuentos por cantidad:

|    | Tramo Pedido   |   Precio Unitario (€) |   Coste Mantener (H) |   Cantidad Pedida (Q) |   Pedidos/año |   Coste Total Anual (€) |
|----|----------------|-----------------------|----------------------|-----------------------|---------------|-------------------------|
|  2 | 2000 - ∞       |                    46 |                  9.2 |                  2327 |        214.06 |             2.29322e+07 |
|  1 | 1000 - 1999    |                    48 |                  9.6 |                  1999 |        249.16 |             2.3929e+07  |
|  0 | 0 - 999        |                    50 |                 10   |                   999 |        498.56 |             2.4933e+07  |

🧠 Optimización continua EOQ con precio mínimo (46 €):

📦 Cantidad óptima a pedir: 2327 uds
🔁 Pedidos por año: 214.04
💸 Coste total anual estimado: 22,932,211.98 €
